# 04. 시계열 예측 (Time-series Forecasting)
## 개인 소비 분석 + 무의식 지출 탐지 AI

**목표:**
- 과거 소비 패턴 기반으로 다음 달 지출 예측
- 요일/주기 패턴 자동 학습
- '이번 달 예산 초과 가능성' 조기 경보

**활용 모델:**
- `Prophet` (Meta) : 트렌드 + 계절성 자동 분해
- `이동평균` : 간단한 베이스라인 비교

## 0. 라이브러리 임포트

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')

# Prophet 설치 확인
try:
    from prophet import Prophet
    PROPHET_OK = True
    print('Prophet 로드 완료!')
except ImportError:
    PROPHET_OK = False
    print('Prophet 미설치 → pip install prophet')
    print('Prophet 없이 이동평균 예측으로 대체합니다.')

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

## 1. 데이터 로드 및 일별 집계

In [ ]:
df = pd.read_csv('../data/processed/spending_with_anomaly.csv')
df['date'] = pd.to_datetime(df['date'])

# 일별 총 지출 집계
daily = (
    df.groupby('date')['amount']
    .sum()
    .reset_index()
    .rename(columns={'date': 'ds', 'amount': 'y'})
)

# 날짜 공백 채우기 (지출 없는 날 = 0)
date_range = pd.date_range(daily['ds'].min(), daily['ds'].max(), freq='D')
daily = daily.set_index('ds').reindex(date_range, fill_value=0).reset_index()
daily.columns = ['ds', 'y']

print(f'기간: {daily["ds"].min().date()} ~ {daily["ds"].max().date()}')
print(f'총 {len(daily)}일 데이터')
daily.tail()

## 2. 데이터 탐색 (예측 전 패턴 확인)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7))

# 일별 지출
axes[0].bar(daily['ds'], daily['y'], color='steelblue', alpha=0.6)
axes[0].plot(daily['ds'], daily['y'].rolling(7, min_periods=1).mean(),
             color='red', linewidth=2, label='7일 이동평균')
axes[0].set_title('일별 지출 (Prophet 입력 데이터)', fontsize=13)
axes[0].set_ylabel('금액 (원)')
axes[0].legend()
axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%m/%d'))

# 요일별 평균 (계절성 힌트)
daily['weekday'] = daily['ds'].dt.dayofweek
wday_avg = daily.groupby('weekday')['y'].mean()
wday_labels = ['월', '화', '수', '목', '금', '토', '일']
colors = ['#FF6B6B' if i >= 5 else '#4ECDC4' for i in range(7)]
axes[1].bar(wday_labels, wday_avg.values, color=colors)
axes[1].set_title('요일별 평균 지출 (주간 계절성)', fontsize=13)
axes[1].set_ylabel('평균 금액 (원)')
for i, v in enumerate(wday_avg.values):
    axes[1].text(i, v + 100, f'{v:,.0f}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('../data/processed/09_forecast_input.png', dpi=120, bbox_inches='tight')
plt.show()

## 3-A. Prophet 예측 (설치된 경우)

In [ ]:
FORECAST_DAYS = 30  # 예측할 미래 일수

if PROPHET_OK:
    # Prophet 모델 설정
    model = Prophet(
        yearly_seasonality=False,   # 1~2달 데이터라 연간 패턴 비활성
        weekly_seasonality=True,    # 요일별 패턴 ON
        daily_seasonality=False,
        changepoint_prior_scale=0.3,  # 트렌드 변화 민감도
        seasonality_mode='additive',
        interval_width=0.80,          # 80% 신뢰구간
    )

    model.fit(daily[['ds', 'y']])

    # 미래 날짜 생성 + 예측
    future = model.make_future_dataframe(periods=FORECAST_DAYS)
    forecast = model.predict(future)

    print(f'예측 완료! (미래 {FORECAST_DAYS}일)')
    forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail(10)
else:
    print('Prophet 미설치 → 3-B 셀(이동평균)로 건너뜁니다.')

In [ ]:
if PROPHET_OK:
    fig, axes = plt.subplots(2, 1, figsize=(14, 9))

    # ① 전체 예측 플롯
    hist = forecast[forecast['ds'] <= daily['ds'].max()]
    pred = forecast[forecast['ds'] > daily['ds'].max()]

    axes[0].bar(daily['ds'], daily['y'], color='steelblue', alpha=0.4, label='실제 지출')
    axes[0].plot(hist['ds'], hist['yhat'], color='orange', linewidth=2, label='모델 적합')
    axes[0].plot(pred['ds'], pred['yhat'], color='red', linewidth=2.5,
                 linestyle='--', label=f'예측 (+{FORECAST_DAYS}일)')
    axes[0].fill_between(pred['ds'], pred['yhat_lower'], pred['yhat_upper'],
                         alpha=0.2, color='red', label='80% 신뢰구간')
    axes[0].axvline(daily['ds'].max(), color='gray', linestyle=':', linewidth=1.5)
    axes[0].set_title('Prophet 소비 예측 (실제 + 미래)', fontsize=13)
    axes[0].set_ylabel('금액 (원)')
    axes[0].legend()
    axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%m/%d'))

    # ② 요일 계절성 분해
    weekly = model.predict(model.make_future_dataframe(periods=7))
    wday_effect = forecast.groupby(forecast['ds'].dt.dayofweek)['weekly'].mean()
    axes[1].bar(wday_labels, wday_effect.values,
                color=['#FF6B6B' if i >= 5 else '#4ECDC4' for i in range(7)])
    axes[1].axhline(0, color='black', linewidth=0.8)
    axes[1].set_title('요일별 소비 계절성 효과 (Prophet 분해)', fontsize=13)
    axes[1].set_ylabel('계절성 효과 (원)')

    plt.tight_layout()
    plt.savefig('../data/processed/10_prophet_forecast.png', dpi=120, bbox_inches='tight')
    plt.show()

## 3-B. 이동평균 예측 (Prophet 미설치 대체)

In [ ]:
if not PROPHET_OK:
    # 최근 7일 이동평균으로 미래 30일 예측
    ma7 = daily['y'].rolling(7, min_periods=1).mean().iloc[-1]
    std7 = daily['y'].rolling(7, min_periods=1).std().iloc[-1]

    last_date = daily['ds'].max()
    future_dates = pd.date_range(last_date + pd.Timedelta(days=1), periods=FORECAST_DAYS)

    # 요일별 가중치 적용
    wday_weight = daily.groupby('weekday')['y'].mean()
    overall_mean = daily['y'].mean()
    wday_ratio = wday_weight / overall_mean

    pred_values = [ma7 * wday_ratio.get(d.dayofweek, 1.0) for d in future_dates]

    forecast_simple = pd.DataFrame({
        'ds': future_dates,
        'yhat': pred_values,
        'yhat_lower': [max(0, v - std7) for v in pred_values],
        'yhat_upper': [v + std7 for v in pred_values],
    })

    fig, ax = plt.subplots(figsize=(14, 5))
    ax.bar(daily['ds'], daily['y'], color='steelblue', alpha=0.5, label='실제 지출')
    ax.plot(forecast_simple['ds'], forecast_simple['yhat'],
            color='red', linewidth=2.5, linestyle='--', label=f'예측 (+{FORECAST_DAYS}일)')
    ax.fill_between(forecast_simple['ds'],
                    forecast_simple['yhat_lower'],
                    forecast_simple['yhat_upper'],
                    alpha=0.2, color='red', label='예측 범위')
    ax.axvline(last_date, color='gray', linestyle=':', linewidth=1.5)
    ax.set_title('이동평균 기반 소비 예측', fontsize=13)
    ax.set_ylabel('금액 (원)')
    ax.legend()
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.savefig('../data/processed/10_ma_forecast.png', dpi=120, bbox_inches='tight')
    plt.show()

    forecast = forecast_simple  # 이후 공통 변수로 사용

## 4. 다음 달 예산 초과 경보

In [ ]:
# 예측 기간의 미래 데이터만 추출
future_only = forecast[forecast['ds'] > daily['ds'].max()]

pred_monthly  = future_only['yhat'].sum()
pred_lower    = future_only['yhat_lower'].sum()
pred_upper    = future_only['yhat_upper'].sum()

# 이번 달 실제 지출
actual_monthly = daily['y'].sum()

# 사용자 예산 설정 (직접 수정 가능)
MONTHLY_BUDGET = actual_monthly * 0.9  # 현재보다 10% 절약 목표

print('=' * 55)
print('  다음 달 소비 예측 리포트')
print('=' * 55)
print(f'  이번 달 실제 지출:   {actual_monthly:>12,.0f}원')
print(f'  다음 달 예측 (중간): {pred_monthly:>12,.0f}원')
print(f'  예측 범위:           {pred_lower:>12,.0f} ~ {pred_upper:,.0f}원')
print(f'  설정 예산:           {MONTHLY_BUDGET:>12,.0f}원')
print('-' * 55)

over = pred_monthly - MONTHLY_BUDGET
if over > 0:
    pct = over / MONTHLY_BUDGET * 100
    print(f'  ⚠️  예산 초과 예상:   +{over:,.0f}원 ({pct:.1f}%)')
    print(f'  → 일평균 {over/30:,.0f}원씩 줄여야 예산 달성 가능')
else:
    print(f'  ✅ 예산 내 예상:      {abs(over):,.0f}원 여유')
print('=' * 55)

## 5. 카테고리별 다음 달 예측

In [ ]:
# 카테고리별 Prophet (또는 이동평균) 예측
cat_forecast = {}

for cat in df['category'].unique():
    cat_daily = (
        df[df['category'] == cat]
        .groupby('date')['amount'].sum()
        .reset_index()
        .rename(columns={'date': 'ds', 'amount': 'y'})
    )
    cat_daily['ds'] = pd.to_datetime(cat_daily['ds'])
    cat_daily = cat_daily.set_index('ds').reindex(date_range, fill_value=0).reset_index()
    cat_daily.columns = ['ds', 'y']

    if PROPHET_OK and cat_daily['y'].sum() > 0:
        try:
            m = Prophet(weekly_seasonality=True, yearly_seasonality=False,
                       daily_seasonality=False, interval_width=0.80)
            m.fit(cat_daily)
            fut = m.make_future_dataframe(periods=30)
            fc = m.predict(fut)
            next_month = fc[fc['ds'] > cat_daily['ds'].max()]['yhat'].sum()
        except:
            next_month = cat_daily['y'].mean() * 30
    else:
        next_month = cat_daily['y'].mean() * 30

    cat_forecast[cat] = max(0, next_month)

cat_fc_df = pd.DataFrame([
    {'카테고리': k,
     '이번달_실제': int(df[df['category'] == k]['amount'].sum()),
     '다음달_예측': int(v)}
    for k, v in cat_forecast.items()
]).sort_values('다음달_예측', ascending=False).reset_index(drop=True)

cat_fc_df['변화율(%)'] = ((cat_fc_df['다음달_예측'] - cat_fc_df['이번달_실제'])
                          / cat_fc_df['이번달_실제'].replace(0, 1) * 100).round(1)

print(cat_fc_df.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

x = np.arange(len(cat_fc_df))
w = 0.35
bars1 = ax.bar(x - w/2, cat_fc_df['이번달_실제'], w, label='이번달 실제', color='#5B8CFF', alpha=0.8)
bars2 = ax.bar(x + w/2, cat_fc_df['다음달_예측'], w, label='다음달 예측', color='#FF6B6B', alpha=0.8)

ax.set_xticks(x)
ax.set_xticklabels(cat_fc_df['카테고리'], rotation=30)
ax.set_ylabel('금액 (원)')
ax.set_title('카테고리별 이번달 vs 다음달 예측 지출', fontsize=13)
ax.legend()

# 변화율 표시
for i, row in cat_fc_df.iterrows():
    color = 'red' if row['변화율(%)'] > 5 else 'green'
    ax.text(i + w/2, row['다음달_예측'] + 500,
            f"{row['변화율(%)']:+.0f}%", ha='center', fontsize=8, color=color)

plt.tight_layout()
plt.savefig('../data/processed/11_category_forecast.png', dpi=120, bbox_inches='tight')
plt.show()

## 6. 최종 요약 리포트

In [ ]:
print('=' * 55)
print('  전체 분석 완료 요약')
print('=' * 55)
print(f'  [EDA]        총 지출 {actual_monthly:,.0f}원')

anomaly_count = df['is_anomaly'].sum() if 'is_anomaly' in df.columns else 0
anomaly_amt   = df[df['is_anomaly'] == True]['amount'].sum() if 'is_anomaly' in df.columns else 0
print(f'  [이상탐지]   이상 소비 {anomaly_count}건 / {anomaly_amt:,.0f}원')
print(f'  [예측]       다음달 예상 {pred_monthly:,.0f}원')
print(f'  [예산경보]   {"⚠️ 초과 예상" if over > 0 else "✅ 예산 내"}')
print('-' * 55)
print('  생성된 분석 파일:')
print('    data/processed/09_forecast_input.png')
print('    data/processed/10_*_forecast.png')
print('    data/processed/11_category_forecast.png')
print('=' * 55)
print()
print('  모든 노트북 완료!')
print('  다음: streamlit run app.py 으로 웹앱 실행')

In [ ]:
# 예측 결과 저장
future_only[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].to_csv(
    '../data/processed/forecast_next30days.csv', index=False, encoding='utf-8-sig'
)
cat_fc_df.to_csv('../data/processed/forecast_by_category.csv', index=False, encoding='utf-8-sig')
print('예측 결과 저장 완료!')